# News Mining & Classification: Visualizations
---
This standalone notebook generates and displays visualizations for:
1. **Class Distribution** (Raw Imbalanced vs. Preprocessed Balanced)
2. **Confusion Matrix Heatmap** (LinearSVC on Held-out Test Set)
3. **Model Performance Metrics** (Precision, Recall, F1-Score per Class)
4. **Latent Semantic Space Projection** (2D PCA of SentenceTransformer Embeddings)
5. **Article Word Count Distribution**

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

# Add root path to import database connection
sys.path.insert(0, str(Path.cwd()))
import scripts.db as db

# Plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

## 1. Class Distribution: Raw vs. Balanced
Visualizing the class imbalance problem in `gold.total_news` (189,512 Real vs. 12,815 Fake) and the balanced dataset (12,815 each) achieved through undersampling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#2b5c8f', '#d95f02']

# Raw distribution
raw_counts = {'Real News (0)': 189512, 'Fake News (1)': 12815}
bars1 = axes[0].bar(raw_counts.keys(), raw_counts.values(), color=colors, width=0.5, edgecolor='black')
axes[0].set_title('Raw Dataset Distribution (Imbalanced)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Articles')
axes[0].set_ylim(0, 210000)
for bar in bars1:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, yval + 3000, f'{yval:,}\n({yval/202327*100:.1f}%)', 
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

# Balanced distribution
bal_counts = {'Real News (0)': 12815, 'Fake News (1)': 12815}
bars2 = axes[1].bar(bal_counts.keys(), bal_counts.values(), color=colors, width=0.5, edgecolor='black')
axes[1].set_title('Preprocessed Dataset Distribution (Balanced)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Articles')
axes[1].set_ylim(0, 16000)
for bar in bars2:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2.0, yval + 300, f'{yval:,}\n(50.0%)', 
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Confusion Matrix Heatmap
Displaying the prediction performance of the trained `LinearSVC` model across the 5,126 test articles.

In [ ]:
cm = np.array([[2514, 49],
               [188, 2375]])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=False, cmap='Blues', fmt='d', cbar=True, ax=ax, linewidths=1, linecolor='gray')

labels = [
    [f"True Real (TN)\n{cm[0,0]:,} ({cm[0,0]/cm[0].sum()*100:.1f}%)", f"False Fake (FP)\n{cm[0,1]:,} ({cm[0,1]/cm[0].sum()*100:.1f}%)"],
    [f"False Real (FN)\n{cm[1,0]:,} ({cm[1,0]/cm[1].sum()*100:.1f}%)", f"True Fake (TP)\n{cm[1,1]:,} ({cm[1,1]/cm[1].sum()*100:.1f}%)"]
]
for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > 1500 else "black"
        ax.text(j + 0.5, i + 0.5, labels[i][j], ha="center", va="center", color=color, fontsize=12, fontweight='bold')

ax.set_title('LinearSVC Confusion Matrix (Test Set: N=5,126)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Predicted Label', fontsize=12, labelpad=8)
ax.set_ylabel('Actual Label', fontsize=12, labelpad=8)
ax.set_xticklabels(['Real News (0)', 'Fake News (1)'], fontsize=11)
ax.set_yticklabels(['Real News (0)', 'Fake News (1)'], fontsize=11, rotation=0)

plt.tight_layout()
plt.show()

## 3. Classification Performance Metrics
Comparing Precision, Recall, and F1-score across both classes.

In [ ]:
metrics_data = {
    'Metric': ['Precision', 'Recall', 'F1-Score'] * 2,
    'Score': [0.93, 0.98, 0.95, 0.98, 0.93, 0.95],
    'Class': ['Real News (0)'] * 3 + ['Fake News (1)'] * 3
}
metrics_df = pd.DataFrame(metrics_data)

fig, ax = plt.subplots(figsize=(9, 5.5))
bar_plot = sns.barplot(data=metrics_df, x='Metric', y='Score', hue='Class', 
                       palette=['#2b5c8f', '#d95f02'], ax=ax, edgecolor='black')
ax.set_title('Model Performance Metrics Comparison by Class (LinearSVC)', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Score (0.0 - 1.0)', fontsize=12)
ax.set_xlabel('Evaluation Metric', fontsize=12)
ax.set_ylim(0, 1.15)
ax.axhline(0.95, color='green', linestyle='--', linewidth=1.2, label='Overall Accuracy (95%)')

for p in bar_plot.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f"{h:.2f}",
                    (p.get_x() + p.get_width() / 2., h),
                    ha='center', va='center',
                    xytext=(0, 7),
                    textcoords='offset points',
                    fontsize=11, fontweight='bold')

ax.legend(title='Category', loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

## 4. Latent Semantic Space (2D PCA of Sentence Transformer Embeddings)
Querying sample records from PostgreSQL `gold.total_news`, encoding them with `sentence-transformers/all-MiniLM-L6-v2`, and projecting the 384-dimensional embeddings to 2D.

In [ ]:
engine = db.db_engine()
query = """
(SELECT article_text, is_fake FROM gold.total_news WHERE is_fake = 0 ORDER BY RANDOM() LIMIT 300)
UNION ALL
(SELECT article_text, is_fake FROM gold.total_news WHERE is_fake = 1 ORDER BY RANDOM() LIMIT 300);
"""
df_sample = pd.read_sql(query, engine)

# Encode using SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
sample_embeddings = model.encode(df_sample['article_text'].tolist(), show_progress_bar=False)

# PCA 2D reduction
pca = PCA(n_components=2, random_state=42)
pca_results = pca.fit_transform(sample_embeddings)
df_sample['PCA1'] = pca_results[:, 0]
df_sample['PCA2'] = pca_results[:, 1]
df_sample['Category'] = df_sample['is_fake'].map({0: 'Real News (0)', 1: 'Fake News (1)'})

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=df_sample,
    x='PCA1',
    y='PCA2',
    hue='Category',
    palette={'Real News (0)': '#2b5c8f', 'Fake News (1)': '#d95f02'},
    alpha=0.75,
    s=50,
    edgecolor='none',
    ax=ax
)
ax.set_title('2D PCA Projection of SentenceTransformer Latent Embeddings', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)', fontsize=11)
ax.set_ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)', fontsize=11)
ax.legend(title='News Class', frameon=True)
plt.tight_layout()
plt.show()

## 5. Article Word Count Distribution
Comparing the length distribution between authentic reporting excerpts and fake news narratives.

In [ ]:
df_sample['word_count'] = df_sample['article_text'].str.split().apply(len)
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    data=df_sample,
    x='word_count',
    hue='Category',
    palette={'Real News (0)': '#2b5c8f', 'Fake News (1)': '#d95f02'},
    bins=30,
    kde=True,
    element='step',
    ax=ax
)
ax.set_title('Word Count Distribution by News Class', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Word Count per Article Description/Text', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
plt.tight_layout()
plt.show()